In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

df = pd.read_csv("WORK_bank_data_train.csv", sep=';')

In [5]:
df

,ID,Age,Ind_Household,Age_group,District,Region,Lifetime,Income,Segment,Ind_deposit,...,Ind_salary,trans_6_month,trans_9_month,trans_12_month,amont_trans,amont_day_from,trans_3_month,Gender,Target1,Target2
0,1200000001,51.0,No,middle,02,Midlands,3.0,53,Platinum,No,...,No,2026.27,2964.23,4140.91,3,21,910.02,F,No,No
1,1200000002,47.0,No,middle,34,Midlands,2.0,51,Gold,No,...,No,2033.14,2969.30,4202.63,5,14,977.80,U,No,No
2,1200000003,45.0,No,middle,17,North,6.0,50,Silver,No,...,No,2085.68,3080.13,4277.45,11,16,1001.54,M,No,No
3,1200000004,78.0,No,senior,49,Midlands,12.0,52,Platinum,Yes,...,No,2026.58,3001.34,4105.54,3,15,980.78,F,No,No
4,1200000005,57.0,No,middle,19,South East,8.0,52,Silver,Yes,...,No,2142.23,3188.19,4401.98,2,26,1003.68,F,Yes,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
985472,1201048571,NaN,No,unknown,12,South East,6.0,42,Silver,Yes,...,No,2139.03,3155.41,4291.97,10,14,989.90,F,No,No
985473,1201048572,63.0,No,senior,33,South East,9.0,51,Tin,No,...,No,1916.91,3000.02,4045.15,11,30,884.63,F,No,No
985474,1201048573,68.0,No,senior,51,North,3.0,56,Gold,Yes,...,No,2053.00,3112.87,4292.20,8,16,922.66,M,No,No
985475,1201048574,45.0,No,middle,45,Midlands,1.0,54,Gold,Yes,...,No,2082.85,3143.77,4207.48,8,15,951.42,M,No,No


In [6]:
df['Target1_bin'] = df['Target1'].map({'Yes': 1, 'No': 0})
df['Target2_bin'] = df['Target2'].map({'Yes': 1, 'No': 0})

exclude_cols = ['ID', 'Target1', 'Target2', 'Target1_bin', 'Target2_bin']
features = [col for col in df.columns if col not in exclude_cols]

In [29]:
def calculate_iv(df, feature, target, bins=10):
    temp = df[[feature, target]].dropna().copy()

    if temp[feature].dtype != 'object':
        temp[feature] = pd.qcut(temp[feature], q=bins, duplicates='drop')
    else:
        temp[feature] = temp[feature].astype(str)

    grouped = temp.groupby(feature)[target].value_counts().unstack().fillna(0)
    if 0 not in grouped.columns or 1 not in grouped.columns:
        return np.nan

    grouped.columns = ['non_event', 'event']
    grouped['event_rate'] = grouped['event'] / grouped['event'].sum()
    grouped['non_event_rate'] = grouped['non_event'] / grouped['non_event'].sum()
    grouped['woe'] = np.log((grouped['event_rate'] + 1e-6) / (grouped['non_event_rate'] + 1e-6))
    grouped['iv'] = (grouped['event_rate'] - grouped['non_event_rate']) * grouped['woe']

    return grouped['iv'].sum()

iv_target1 = {f: calculate_iv(df, f, 'Target1_bin') for f in features}
iv_target2 = {f: calculate_iv(df, f, 'Target2_bin') for f in features}

iv_df = pd.DataFrame({
    'Feature': features,
    'IV_Target1': [iv_target1[f] for f in features],
    'IV_Target2': [iv_target2[f] for f in features]
}).sort_values(by='IV_Target1', ascending=False)

strong_iv = iv_df[(iv_df['IV_Target1'] > 0.015) | (iv_df['IV_Target2'] > 0.015)]

/var/folders/_k/vxqnb8117qb_1tpg9p98mtyc0000gn/T/ipykernel_22049/34899181.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = temp.groupby(feature)[target].value_counts().unstack().fillna(0)
/var/folders/_k/vxqnb8117qb_1tpg9p98mtyc0000gn/T/ipykernel_22049/34899181.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = temp.groupby(feature)[target].value_counts().unstack().fillna(0)
/var/folders/_k/vxqnb8117qb_1tpg9p98mtyc0000gn/T/ipykernel_22049/34899181.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=Fa

In [30]:
strong_iv

,Feature,IV_Target1,IV_Target2
16,amont_day_from,1.559889,3.486207
0,Age,0.691284,0.430057
18,Gender,0.431154,0.301043
2,Age_group,0.193862,0.130816
6,Income,0.074162,0.049218
7,Segment,0.069175,0.046705
3,District,0.050321,0.034477
5,Lifetime,0.029486,0.020546
17,trans_3_month,0.001274,0.015315
